In [ ]:
# import necessary packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from energy_forecast.notebook_setup import DATA
from energy_forecast.io_utils import save_file

In [ ]:
# load energy and weather datasets into pandas dataframes
energy_data = pd.read_csv(DATA / 'energy_dataset.csv')
weather_data = pd.read_csv(DATA / 'weather_features.csv')

In [ ]:
energy_data.info()

In [ ]:
energy_data['time'] = pd.to_datetime(energy_data['time'], yearfirst=True, utc=True)

In [ ]:
energy_data['generation hydro pumped storage consumption'].describe()

In [ ]:
energy_data.info()

In [ ]:
#it looks like we have several columns that are completely null. Let's confirm and then delete them:
energy_data['generation fossil coal-derived gas'].sum()

In [ ]:
energy_data['generation fossil oil shale'].sum()

In [ ]:
energy_data['generation fossil peat'].sum()

In [ ]:
energy_data['generation hydro pumped storage aggregated'].count()

In [ ]:
energy_data['generation geothermal'].sum()

In [ ]:
energy_data['generation marine'].sum()

In [ ]:
energy_data['generation wind offshore'].sum()

In [ ]:
energy_data['forecast wind offshore eday ahead'].count()

In [ ]:
energy_data.drop(columns = ['generation fossil coal-derived gas','generation fossil oil shale','generation fossil peat',
                            'generation geothermal','generation hydro pumped storage aggregated','generation marine',
                            'generation wind offshore','forecast wind offshore eday ahead'],inplace=True)

In [ ]:
#Count (using `.sum()`) the number of missing values (`.isnull()`) in each column of 
#energy_data as well as the percentages (using `.mean()` instead of `.sum()`).
#Order them (increasing or decreasing) using sort_values
#Call `pd.concat` to present these in a single table (DataFrame) with the helpful column names 'count' and '%'
missing = pd.concat([energy_data.isnull().sum(), 100 * energy_data.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by=['count'],ascending = False)
missing.head(20)


The number of null values is very low, but I want to delete as few rows as possible because it would be best to have continuous hourly data. I could impute the missing values with 0 and it would likely not have a significant effect. It is likely that the missing values all come from the same times, since the number of missing values is very similar for each variable. I will need to investigate how best to handle these.

In [ ]:
# let's check to see what the null values look like for our target variable:
energy_data.query('`total load actual`.isnull()', engine='python') 

For the most part, I just have single hours of missing 'total load actual' here and there, except for two instances where there a substantial period of many null vlaues in a row (on 2015-01-05 and 2015-02-01)

In [ ]:
energy_data.info()

As suspected, the rows that are null are pretty much null across the board. Luckily most of them are not continuous, except for one group of 6 continuous hours on 2015-01-05. Probably the only way to handle these null values is to delete the entire row since it would not be feasible to impute missing values for all the columns of an observation. So if it is a row with all 'generation' columns as null, I will delete the row, but if only a few columns are null, I will impute missing vlaues with 0. But it is maybe best not to delete these until we merge with the weather data.

In [ ]:
weather_data.info
#we can see that the times seem to match up, so if I want to merge the two DF it will be on 'time' = 'dt_iso',
#however, there is only one instance of each time in the energy data and one for each city (x5) in the weather data
#It also appears that there is indeed one observation for each hour over four years. 
#365*3*24 + 366*24 = 35064 (for one leap year). Somehow the weather data has more entries than that though. 
# 5 cities * 35064 = 175320, but there are 178395 rows. I will have to invesigate where the ~3000 extra rows come from

In [ ]:
weather_data[weather_data.city_name == 'Valencia'].count()

In [ ]:
weather_data[weather_data.city_name == 'Madrid'].count()

In [ ]:
weather_data[weather_data.city_name == 'Bilbao'].count()

In [ ]:
weather_data[weather_data.city_name == ' Barcelona'].count() #noticed that Barecelona has a space in front

In [ ]:
weather_data[weather_data.city_name == 'Seville'].count()

In [ ]:
#delete the space in the Barcelona city name before I forget
weather_data.replace(to_replace =' Barcelona', value ='Barcelona', inplace = True)

In [ ]:
weather_data['dt_iso'].value_counts()

So there are many actual duplicates (not just the 5x duplicates indicated by having 5 cities). We will eventually need to transform the weather data to be wide (have only one time observation per hour and make the city data be columns).

In [ ]:
weather_data.drop_duplicates(inplace = True) 

In [ ]:
weather_data['dt_iso'].value_counts()

In [ ]:
#dropping duplicates did not solve the issue so they must not be duplicates in every field
#let's see what one of the duplicate time values rows look like:
weather_data[weather_data.dt_iso == '2018-02-28 09:00:00+01:00']

So we have instances where there are multiple hourly observations for the same city, but with different descriptors, id, icon, etc. I'd like to delete all but the first hourly observation for each city. 

In [ ]:
#I suspect that temp, temp_min, and temp_max are all equal, let's check and delete the two extra columns if so.
weather_data["temp"].equals(weather_data["temp_min"])
#Okay, columns are not always the same, but since the data are hourly I assume it will be safe to delete 
# temp_max and temp_min and just use temp

In [ ]:
weather_data["weather_description"].unique()
#These seem too detailed and too numerous to be of use; this column should likely be deleted

In [ ]:
weather_data["weather_id"].unique()
# these are codes that go along with the descriptors above

In [ ]:
weather_data["weather_main"].unique()
#This variable seems like it is basic enough that it would be worth quantifying to make a numerical categorical variable

In [ ]:
weather_data["weather_icon"].unique()
#these are codes for the weather icon (or thumbnail picture) to describe weather (ie picture of sun or clouds, etc)

In [ ]:
#let's have a more detailed look at the rain, snow, and clouds features
weather_data[['rain_1h','rain_3h', 'snow_3h', 'clouds_all']].describe()

It appears the rain_3h mean and max are less than the rain_1h mean and max, which shouldn't be true, so there may be something funky going on. And in any case, 1hr rainfall data should be sufficient. So let's delete the 3h column. I'll also delete the 'weather_id','weather_icon','weather_description','temp_min', and 'temp_max' columns.

In [ ]:
#I suspect I will also not need rain_1hr or snow_3hr, and maybe not wind_deg, but I will keep them for now.
weather_data.drop(columns = ['rain_3h', 'weather_id','weather_icon','weather_description','temp_min','temp_max'],inplace = True)

In [ ]:
#let's delete duplicate rows now:
weather_data.drop_duplicates(subset = ['dt_iso','city_name'], keep='first', inplace=True)

In [ ]:
weather_data[weather_data.city_name == 'Valencia'].count()

In [ ]:
weather_data[weather_data.city_name == 'Madrid'].count()

In [ ]:
weather_data[weather_data.city_name == 'Barcelona'].count()

In [ ]:
weather_data[weather_data.city_name == 'Bilbao'].count()

In [ ]:
weather_data[weather_data.city_name == 'Seville'].count()

Excellent, now there is only one hourly observation for each city. Now I have to determine what would be the best way to merge the energy and weather dataframes. If I just average the columns in the weather data for all five cities I might lose some meaningful information, but there would be only one neat set of weather variables. I could also create a wide dataframe with columns for each of the weather variables for each city, but this may create too many variables. Not sure which approach is better...

In [ ]:
# create a df with averages for each column (drop city_name column)
weather_average = weather_data.drop(columns = ['city_name','weather_main', 'wind_deg']) #deleting wind deg also because it is
                                                                            #categorical but not useful enough to encode
#create dict for weather_main:numeric value
#weather_dict = {'clear':0, 'clouds':1, 'rain':2, 'mist':3, 'thunderstorm':4, 'drizzle':5, 
#                'fog':6, 'smoke':7, 'haze':8, 'snow':9, 'dust':10, 'squall':11}

# categorize the weather_main column 
#weather_avg['weather_main'].replace(weather_dict, inplace = True)

#making sure the value remap worked:
#weather_avg['weather_main'].value_counts()

# average all other columns grouped by time
weather_avg = weather_average.groupby(['dt_iso'],as_index = False).mean()  #categorical 'weather_main' average will be distorted, take floor int value to convert back to category? 
weather_avg.head()


Excellent, it appears it worked to take the averages of the variables for the 5 cities to have only one observation per hour! I'm not sure what the best method for encoding the nominal weather_main variable is. After first Label Encoding them, I have decided for now to drop this column from the averaged df. It seems there are too many categories to do One-Hot encoding and label encoding would not be appropriate for a nominal variable. For now I will try working without this variable, but I could do some feature engineering later to create an ordinal categorical variable with fewer categories.

In [ ]:
#create a wide df with the five cities converted to columns and only one time observation per hour

weather_wide = weather_data.pivot(index = 'dt_iso',columns = 'city_name', values = ['temp','pressure','humidity','wind_speed',
'wind_deg','rain_1h','snow_3h','clouds_all','weather_main'])
weather_wide.head()

In [ ]:
weather_wide.reset_index(inplace = True)
weather_wide.head(2)

In [ ]:
weather_wide.columns = weather_wide.columns.map('_'.join)
weather_wide.head()

In [ ]:
weather_wide.drop(columns = ['weather_main_Barcelona','weather_main_Bilbao','weather_main_Madrid','weather_main_Seville','weather_main_Valencia'],inplace = True)
weather_wide.head()

In [ ]:
# save the data to a new csv file
datapath = DATA
save_file(weather_wide, 'weather_wide.csv', datapath)

In [ ]:
# save the data to a new csv file
datapath = DATA
save_file(weather_avg, 'weather_avg.csv', datapath)

In [ ]:
datapath = DATA
save_file(energy_data, 'energy_data.csv', datapath)